In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 와이어 컷을 사용한 Circuit 커팅 시작하기


<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시기를 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
qiskit-addon-cutting~=0.10.0
```
</details>
이 가이드에서는 `qiskit-addon-cutting` 패키지를 사용한 와이어 컷의 동작 예제를 시연합니다. 와이어 커팅을 사용하여 7-Qubit Circuit의 기댓값을 재구성하는 방법을 다룹니다.

이 패키지에서 와이어 컷은 2-Qubit [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) 명령어로 표현됩니다. 이 명령어는 두 번째 Qubit의 리셋 후 두 Qubit의 스왑으로 정의됩니다. 이 연산은 첫 번째 Qubit의 상태를 두 번째 Qubit으로 전달하는 동시에 두 번째 Qubit의 입력 상태를 버리는 것과 동일합니다.

이 패키지는 물리적 Qubit에 작용할 때 와이어 컷을 처리해야 하는 방식과 일관되도록 설계되었습니다. 예를 들어, 와이어 컷은 물리적 Qubit $n$의 상태를 취하고 컷 이후 물리적 Qubit $m$으로 계속할 수 있습니다. "명령어 커팅"은 와이어 컷과 Gate 컷 모두를 동일한 형식 내에서 고려하기 위한 통합 프레임워크로 생각할 수 있습니다(와이어 컷은 단지 컷된 [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) 명령어이기 때문입니다). 와이어 커팅에 이 프레임워크를 사용하면 Qubit 재사용도 가능하며, 이는 [수동으로 와이어 커팅하기](#cut-wires-using-the-low-level-move-instruction) 섹션에서 설명됩니다.

단일 Qubit [`CutWire`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-cut-wire) 명령어는 와이어 컷 작업을 위한 보다 추상화된 간단한 인터페이스 역할을 합니다. 이를 통해 높은 수준에서 Circuit의 어디에 와이어를 절단할지 표시하고 Circuit 커팅 애드온이 적절한 [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) 명령어를 삽입하도록 할 수 있습니다.

다음 예제는 와이어 커팅 후 기댓값 재구성을 시연합니다. 여러 비로컬 Gate가 있는 Circuit을 생성하고 추정할 관찰 가능량을 정의합니다.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit_ibm_runtime import SamplerV2, Batch
from qiskit_aer.primitives import EstimatorV2

from qiskit_addon_cutting.instructions import Move, CutWire
from qiskit_addon_cutting import (
    partition_problem,
    generate_cutting_experiments,
    cut_wires,
    expand_observables,
    reconstruct_expectation_values,
)


qc_0 = QuantumCircuit(7)
for i in range(7):
    qc_0.rx(np.pi / 4, i)
qc_0.cx(0, 3)
qc_0.cx(1, 3)
qc_0.cx(2, 3)
qc_0.cx(3, 4)
qc_0.cx(3, 5)
qc_0.cx(3, 6)
qc_0.cx(0, 3)
qc_0.cx(1, 3)
qc_0.cx(2, 3)

# Define observable
observable = SparsePauliOp(["ZIIIIII", "IIIZIII", "IIIIIIZ"])

# Draw circuit
qc_0.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/b481ef2d-3912-4eac-9755-335e8f5db886-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/b481ef2d-3912-4eac-9755-335e8f5db886-0.svg)

## 고수준 `CutWire` 명령어를 사용하여 와이어 커팅하기
다음으로, Qubit $q_3$에 단일 Qubit [`CutWire`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-cut-wire) 명령어를 사용하여 와이어 컷을 만듭니다. 하위 실험이 실행 준비가 되면, [`cut_wires()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#cut_wires) 함수를 사용하여 `CutWire`를 새로 할당된 Qubit의 [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) 명령어로 변환합니다.

In [2]:
qc_1 = QuantumCircuit(7)
for i in range(7):
    qc_1.rx(np.pi / 4, i)
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)
qc_1.append(CutWire(), [3])
qc_1.cx(3, 4)
qc_1.cx(3, 5)
qc_1.cx(3, 6)
qc_1.append(CutWire(), [3])
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)

qc_1.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/9bac1915-316b-49d0-a1a1-145047679530-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/9bac1915-316b-49d0-a1a1-145047679530-0.svg)

> **Info:** Circuit이 하나 이상의 와이어 컷을 통해 확장될 때, 관찰 가능량은 도입된 추가 Qubit을 반영하도록 업데이트되어야 합니다. `qiskit-addon-cutting` 패키지에는 편의 함수 [`expand_observables()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#expand_observables)가 있으며, `PauliList` 객체와 원본 및 확장된 Circuit을 인수로 받아 새로운 `PauliList`를 반환합니다.
> 
>     반환된 `PauliList`에는 원본 관찰 가능량의 계수에 대한 정보가 포함되지 않지만, 이는 최종 기댓값 재구성까지 무시할 수 있습니다.

In [3]:
# Transform CutWire instructions to Move instructions
qc_2 = cut_wires(qc_1)

# Expand the observable to match the new circuit size
expanded_observable = expand_observables(observable.paulis, qc_0, qc_2)
print(f"Expanded Observable: {expanded_observable}")
qc_2.draw("mpl")

Expanded Observable: ['ZIIIIIIII', 'IIIZIIIII', 'IIIIIIIIZ']


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d398b397-0167-4db9-97ae-6ea502dc43e3-1.svg" alt="Output of the previous code cell" />

### Partition the circuit and observable

Now the problem can be separated into partitions. This is accomplished using the [`partition_problem()`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#partition_problem) function with an optional set of partition labels to specify how to separate the circuit. Qubits sharing a common partition label are grouped together, and any non-local gates spanning more than one partition are cut.

If no partition labels are provided, then the partitioning will be automatically determined based on the connectivity of the circuit. Read the next section on [cutting wires manually](#cut-wires-using-the-low-level-move-instruction) for more information on including partition labels.

In [4]:
partitioned_problem = partition_problem(
    circuit=qc_2,
    observables=expanded_observable,
)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables
bases = partitioned_problem.bases

print(f"Subobservables to measure: \n{subobservables}\n")
print(f"Sampling overhead: {np.prod([basis.overhead for basis in bases])}")
subcircuits[0].draw("mpl")

Subobservables to measure: 
{0: PauliList(['IIIII', 'ZIIII', 'IIIIZ']), 1: PauliList(['ZIII', 'IIII', 'IIII'])}

Sampling overhead: 256.0


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/5fb034f2-da8a-4f4d-ab9b-c990593e04fc-1.svg" alt="Output of the previous code cell" />

In [5]:
subcircuits[1].draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d0e86f81-7c7e-4ccf-951c-9cd039135dc9-0.svg" alt="Output of the previous code cell" />

In this partitioning scheme, you have cut two wires, resulting in a sampling overhead of $4^4$.

### Generate subexperiments to execute and post-process results

To estimate the expectation value of the full-sized circuit, several subexperiments are generated from the decomposed gates' joint quasi-probability distribution and then executed on one (or more) QPUs. The [`generate_cutting_experiments`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) method does this by ingesting arguments for the `subcircuits` and `subobservables` dictionaries you created above, as well as for the number of samples to take from the distribution.

<Admonition type="note" title="Note about the number of samples">
    The `num_samples` argument specifies how many samples to draw from the quasi-probability distribution and determines the accuracy of the coefficients used for the reconstruction. Passing infinity (`np.inf`) ensures all coefficients are calculated exactly. Read the API docs on [generating weights](/docs/api/qiskit-addon-cutting/qpd#generate_qpd_weights) and [generating cutting experiments](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) for more information.
</Admonition>

In [6]:
# Generate subexperiments
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits, observables=subobservables, num_samples=np.inf
)

# Set a backend to use and transpile the subexperiments
backend = FakeManilaV2()
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
isa_subexperiments = {
    label: pass_manager.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Submit each partition's subexperiments to the Qiskit Runtime Sampler
# primitive, in a single batch so that the jobs will run back-to-back.
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
    # Retrieve results
    results = {label: job.result() for label, job in jobs.items()}

Lastly, the expectation value of the full circuit can be reconstructed using the [`reconstruct_expectation_values()`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#reconstruct_expectation_values) method.


The code block below reconstructs the results and compares them with the exact expectation value.

In [7]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
# Apply the coefficients of the original observable
reconstructed_expval = np.dot(reconstructed_expval_terms, observable.coeffs)


# Compute the exact expectation value using the `qiskit_aer` package.
estimator = EstimatorV2()
exact_expval = estimator.run([(qc_0, observable)]).result()[0].data.evs
print(
    f"Reconstructed expectation value: {np.real(np.round(reconstructed_expval, 8))}"
)
print(f"Exact expectation value: {np.round(exact_expval, 8)}")
print(
    f"Error in estimation: {np.real(np.round(reconstructed_expval-exact_expval, 8))}"
)
print(
    f"Relative error in estimation: {np.real(np.round((reconstructed_expval-exact_expval) / exact_expval, 8))}"
)

Reconstructed expectation value: 1.45965266
Exact expectation value: 1.59099026
Error in estimation: -0.1313376
Relative error in estimation: -0.08255085


![이전 코드 셀의 출력](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d0e86f81-7c7e-4ccf-951c-9cd039135dc9-0.svg)

이 파티션 방식에서는 두 개의 와이어를 절단했으며, 그 결과 $4^4$의 샘플링 오버헤드가 발생합니다.

### 실행할 하위 실험 생성 및 결과 후처리

전체 크기 Circuit의 기댓값을 추정하기 위해, 분해된 Gate의 결합 준확률 분포에서 여러 하위 실험이 생성된 다음 하나 이상의 QPU에서 실행됩니다. [`generate_cutting_experiments`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) 메서드는 위에서 생성한 `subcircuits` 및 `subobservables` 딕셔너리에 대한 인수와 분포에서 추출할 샘플 수에 대한 인수를 받아 이 작업을 수행합니다.

> **Note:** `num_samples` 인수는 준확률 분포에서 추출할 샘플 수를 지정하며 재구성에 사용되는 계수의 정확도를 결정합니다. 무한대(`np.inf`)를 전달하면 모든 계수가 정확하게 계산됩니다. 자세한 내용은 [가중치 생성](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qpd#generate_qpd_weights) 및 [커팅 실험 생성](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments)에 관한 API 문서를 읽어보세요.

In [8]:
qc_1 = QuantumCircuit(8)
for i in [*range(4), *range(5, 8)]:
    qc_1.rx(np.pi / 4, i)
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)
qc_1.append(Move(), [3, 4])
qc_1.cx(4, 5)
qc_1.cx(4, 6)
qc_1.cx(4, 7)
qc_1.append(Move(), [4, 3])
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)

# Expand observable
observable_expanded = SparsePauliOp(["ZIIIIIII", "IIIIZIII", "IIIIIIIZ"])
qc_1.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/15461a2c-85a9-4cb2-a632-b9597ccbc4bd-0.svg" alt="Output of the previous code cell" />

마지막으로, [`reconstruct_expectation_values()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#reconstruct_expectation_values) 메서드를 사용하여 전체 Circuit의 기댓값을 재구성할 수 있습니다.

아래 코드 블록은 결과를 재구성하고 정확한 기댓값과 비교합니다.

In [9]:
partitioned_problem = partition_problem(
    circuit=qc_1,
    partition_labels="AAAABBBB",
    observables=observable_expanded.paulis,
)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables
bases = partitioned_problem.bases

print(f"Subobservables to measure: \n{subobservables}\n")
print(f"Sampling overhead: {np.prod([basis.overhead for basis in bases])}")
subcircuits["A"].draw("mpl")

Subobservables to measure: 
{'A': PauliList(['IIII', 'ZIII', 'IIIZ']), 'B': PauliList(['ZIII', 'IIII', 'IIII'])}

Sampling overhead: 256.0


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/2139745a-bdc3-40bd-bd6f-d26d2a5b5b14-1.svg" alt="Output of the previous code cell" />

In [10]:
subcircuits["B"].draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/4aeb3f1f-a55e-49c4-a7bd-837132429ee1-0.svg" alt="Output of the previous code cell" />

> **Caution:** 기댓값을 정확하게 재구성하려면, 커팅 실험이 생성되거나 관찰 가능량이 확장될 때 이 정보가 손실되었기 때문에 원본 관찰 가능량의 계수(`generate_cutting_experiments()`의 출력과 다름)를 재구성 출력에 적용해야 합니다.
> 
>     일반적으로 이러한 계수는 이전에 표시된 것처럼 `numpy.dot()`를 통해 적용할 수 있습니다.
## 저수준 `Move` 명령어를 사용하여 와이어 커팅하기
고수준 `CutWire` 명령어 사용의 한 가지 제한 사항은 Qubit 재사용을 허용하지 않는다는 것입니다. 커팅 실험에서 이것이 필요한 경우, [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) 명령어를 수동으로 배치할 수 있습니다. 그러나 `Move` 명령어는 대상 Qubit의 상태를 버리기 때문에, 이 Qubit이 나머지 시스템과 얽힘을 공유하지 않는 것이 중요합니다. 그렇지 않으면 리셋 연산으로 인해 와이어 컷 후 Circuit의 상태가 부분적으로 붕괴됩니다.

아래 코드 블록은 이전에 표시된 것과 동일한 예제 Circuit에서 Qubit $q_3$에 대한 와이어 컷을 수행합니다. 여기서의 차이점은 두 번째 와이어 컷이 이루어진 곳에서 `Move` 연산을 역전시켜 Qubit을 재사용할 수 있다는 것입니다(그러나 이것이 항상 가능한 것은 아니며 절단되는 Circuit에 따라 다릅니다).